# Model: will the runner be safe?

When a single goes to the outfield with a runner on second, the third-base coach has to decide: **send** the runner home or **hold** him at third. We call each of these plays an *opportunity*.

This notebook builds a model that looks at an opportunity and estimates **P(safe)**, the chance the runner scores if he is sent. The model learns from 2022–2025 sends, where we know whether the runner was safe or out, and then predicts every 2026 opportunity. It never sees 2026 while learning, so its 2026 predictions are a fair test.

## Setup

`FEATURES` are the facts about each play that the model gets to see:

| Feature | What it means |
|---|---|
| `launch_speed` | Exit velocity: how hard the ball was hit, in mph |
| `launch_angle` | How steeply the ball left the bat, in degrees (negative means down into the ground) |
| `spray_angle` | Direction: about −30 is left field, 0 is straight up the middle, +30 is right field |
| `outs_when_up` | Outs before the play: 0, 1 or 2 |
| `runner_speed` | The runner's sprint speed from Baseball Savant, in feet per second |
| `fielder_arm` | The outfielder's arm strength from Baseball Savant, in mph |
| `bb_type` | Ground ball, line drive, fly ball or popup |
| `hit_location` | Which outfielder fielded it: 7 = left, 8 = center, 9 = right |

If a player has no Savant number for a season, the middle (median) value for that season is used instead.

In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

TEST_SEASON = 2026
FEATURES = ["launch_speed", "launch_angle", "spray_angle", "outs_when_up",
            "runner_speed", "fielder_arm", "bb_type", "hit_location"]

## Load the plays

Each row of `opportunities.csv` is one opportunity. A few plays are missing exit velocity, launch angle or direction, so only `complete` plays are kept.

The model needs numbers, but `bb_type` and `hit_location` are labels. `get_dummies` turns each label into its own yes/no column of 1s and 0s, like `bb_type_line_drive`.

- `train`: 2022–2025 plays where the runner was sent. These are the only plays where we know the answer, safe or out.
- `test`: every 2026 play, sent or held.
- `fly_or_liner`: fly balls and line drives, used for a second model that checks the first.

In [ ]:
plays = pd.read_csv("../opportunities.csv")
plays = plays[plays.complete]

X = pd.get_dummies(plays[FEATURES], columns=["bb_type", "hit_location"], dtype=float)

train = plays.sent & (plays.season < TEST_SEASON)
test = plays.season == TEST_SEASON
fly_or_liner = plays.bb_type.isin(["fly_ball", "line_drive"])

## Train the model and predict 2026

The model is a *gradient boosting* model. It builds a long chain of small decision trees. Each tree is a couple of yes/no questions, like "Was the ball hit harder than 95 mph?" or "Is the runner faster than 28 ft/s?", that end in a guess. Each new tree focuses on fixing the mistakes of the trees before it, and the final answer combines all of them.

`max_depth=2` limits each tree to two questions in a row. Outs are rare, only about 350 in four seasons, so bigger trees would start memorizing flukes instead of learning real patterns. Everything else uses scikit-learn's standard settings.

`predict_proba` gives two chances for each play, out and safe. `[:, 1]` keeps the chance of safe.

The second model learns from fly balls and line drives only. On those plays, where the ball lands is closest to where the outfielder picks it up, so it checks that the main results hold up.

In [ ]:
def train_and_predict(train_rows):
    model = HistGradientBoostingClassifier(max_depth=2, random_state=0)
    model.fit(X[train_rows], plays.safe[train_rows].astype(int))
    return model.predict_proba(X[test])[:, 1]


plays.loc[test, "p_safe"] = train_and_predict(train)
plays.loc[test, "p_safe_fly_liner"] = train_and_predict(train & fly_or_liner)

## Save

Every 2026 play now has `p_safe` from the main model and `p_safe_fly_liner` from the fly-ball and line-drive model. Plays before 2026 are left blank, because the model learned from them.

In [ ]:
plays.to_csv("../scored.csv", index=False)